In [ ]:
from pathlib import Path
import subprocess
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(project_root / 'requirements.txt')])


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
print(f'Seed fixed at {SEED}')
print(f'Project root: {PROJECT_ROOT}')


# Notebook 05 ? CASA-Net Training + Multi-Hazard Extension
## Trains flood CASA-Net and scaffolds shared-backbone multi-hazard learning


## Section 5.1 — Reproducibility Setup

The ablation only means something if every variant runs under the same seed, optimizer, scheduler, and split definition. This setup also auto-reduces the batch size when GPU is unavailable or a `--cpu` flag is present.


In [ ]:
import json
import math
import os
import random
import sys
from copy import deepcopy
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_curve, auc
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    import segmentation_models_pytorch as smp
except Exception:
    smp = None

PROCESSED_PATCHES = PROCESSED_DIR / 'patches'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
REPORT_DIR = OUTPUTS_DIR / 'report'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
FORCE_CPU = '--cpu' in sys.argv or os.getenv('CASA_FORCE_CPU', '0') == '1'
DEVICE = torch.device('cpu' if FORCE_CPU or not torch.cuda.is_available() else 'cuda')
BATCH_SIZE = 4 if DEVICE.type == 'cpu' else 8
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'SMP: {getattr(smp, "__version__", "not-installed")}')
print(f'Batch size: {BATCH_SIZE}')


## Section 5.2 — Dataset Class

The dataset class loads the saved patch archives, computes normalization statistics from the training split only, and applies identical spatial augmentation to VV, VH, terrain, and mask. This keeps the ablation study fair and leakage-free.


In [ ]:
SPLIT_INDEX_PATH = PROCESSED_PATCHES / 'split_index.json'
NORM_STATS_PATH = PROCESSED_PATCHES / 'norm_stats.json'

def normalize(arr, mean, std):
    return (arr - mean) / max(std, 1e-6)

def normalize_terrain(arr):
    arr = arr.astype(np.float32)
    out = np.zeros_like(arr, dtype=np.float32)
    for idx in range(arr.shape[0]):
        channel = arr[idx]
        mn = channel.min()
        mx = channel.max()
        if mx > mn:
            out[idx] = (channel - mn) / (mx - mn)
    return out

class SylhetFloodDataset(Dataset):
    def __init__(self, split='train', split_index_path=SPLIT_INDEX_PATH, augment=True):
        self.split = split
        self.augment = augment
        self.split_index = json.loads(Path(split_index_path).read_text())
        self.patch_paths = [Path(p) for p in self.split_index[split]]
        self.norm_stats = json.loads(NORM_STATS_PATH.read_text()) if NORM_STATS_PATH.exists() else self.compute_normalization_stats()
        self.vv_mean = self.norm_stats['vv_mean']
        self.vv_std = self.norm_stats['vv_std']
        self.vh_mean = self.norm_stats['vh_mean']
        self.vh_std = self.norm_stats['vh_std']
        self.spatial_aug = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5)], additional_targets={'vh': 'image', 'terrain_0': 'image', 'terrain_1': 'image', 'terrain_2': 'image', 'terrain_3': 'image', 'mask': 'mask'})

    def __len__(self):
        return len(self.patch_paths)

    def __getitem__(self, idx):
        npz = np.load(self.patch_paths[idx])
        vv = normalize(npz['vv'].astype(np.float32), self.vv_mean, self.vv_std)
        vh = normalize(npz['vh'].astype(np.float32), self.vh_mean, self.vh_std)
        terrain = normalize_terrain(npz['terrain'])
        mask = npz['mask'].astype(np.float32)
        if self.augment and self.split == 'train':
            aug = self.spatial_aug(image=vv[0], vh=vh[0], terrain_0=terrain[0], terrain_1=terrain[1], terrain_2=terrain[2], terrain_3=terrain[3], mask=mask[0])
            vv = aug['image'][None, ...] + np.random.normal(0.0, 0.05, size=vv.shape).astype(np.float32)
            vh = aug['vh'][None, ...] + np.random.normal(0.0, 0.05, size=vh.shape).astype(np.float32)
            terrain = np.stack([aug['terrain_0'], aug['terrain_1'], aug['terrain_2'], aug['terrain_3']], axis=0)
            mask = aug['mask'][None, ...]
        return torch.tensor(vv), torch.tensor(vh), torch.tensor(terrain), torch.tensor(mask)

    def compute_normalization_stats(self):
        vv_vals, vh_vals = [], []
        for patch_path in tqdm([Path(p) for p in self.split_index['train']], desc='Computing normalization stats'):
            npz = np.load(patch_path)
            vv_vals.append(npz['vv'].ravel())
            vh_vals.append(npz['vh'].ravel())
        stats = {'vv_mean': float(np.concatenate(vv_vals).mean()), 'vv_std': float(np.concatenate(vv_vals).std() + 1e-6), 'vh_mean': float(np.concatenate(vh_vals).mean()), 'vh_std': float(np.concatenate(vh_vals).std() + 1e-6)}
        NORM_STATS_PATH.write_text(json.dumps(stats, indent=2))
        return stats

train_ds = SylhetFloodDataset(split='train', augment=True)
val_ds = SylhetFloodDataset(split='val', augment=False)
test_ds = SylhetFloodDataset(split='test', augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train/Val/Test sizes: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')


## Section 5.3.1 — CPAG (Cross-Polarization Attention Gate)

CPAG lets VH features highlight flooded vegetation and paddy-field responses that VV alone may miss, while VV helps suppress false positives from noisy textured areas and urban double-bounce signatures.


In [ ]:
class CPAG(nn.Module):
    def __init__(self, vv_channels, vh_channels, inter_channels):
        super().__init__()
        self.Q = nn.Conv2d(vv_channels, inter_channels, 1)
        self.K = nn.Conv2d(vh_channels, inter_channels, 1)
        self.V = nn.Conv2d(vh_channels, vh_channels, 1)
        self.proj = nn.Conv2d(vh_channels, vv_channels, 1)
        self.scale = inter_channels ** -0.5

    def forward(self, f_vv, g_vh):
        if g_vh.shape[-2:] != f_vv.shape[-2:]:
            g_vh = F.interpolate(g_vh, size=f_vv.shape[-2:], mode='bilinear', align_corners=False)
        q = self.Q(f_vv).flatten(2)
        k = self.K(g_vh).flatten(2)
        v = self.V(g_vh).flatten(2)
        b, _, hw = q.shape
        side = int(hw ** 0.5)
        attn = torch.softmax(torch.bmm(q.transpose(1, 2), k) * self.scale, dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2)).reshape(b, -1, side, side)
        return f_vv + self.proj(out)


## Section 5.3.2 — HaorTerrainBranch

The terrain branch encodes slope, TWI, HAND, and permanent water into a compact context vector. FiLM modulation then conditions each decoder stage on that terrain context, which is critical for haor wetland disambiguation.


In [ ]:
def apply_film(x, film_params):
    gamma, beta = film_params.chunk(2, dim=1)
    return gamma.unsqueeze(-1).unsqueeze(-1) * x + beta.unsqueeze(-1).unsqueeze(-1)

class HaorTerrainBranch(nn.Module):
    def __init__(self, in_channels=4, decoder_channels=(256, 128, 64, 32)):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.film_generators = nn.ModuleList([nn.Linear(64, 2 * ch) for ch in decoder_channels])

    def forward(self, terrain_input):
        context = self.encoder(terrain_input).flatten(1)
        return [layer(context) for layer in self.film_generators]


## Section 5.3.3 — UncertaintyHead + `mc_inference()`

MC Dropout turns the segmentation head into an uncertainty-aware predictor. The mean output becomes the flood-probability map, and the standard deviation across passes becomes the epistemic uncertainty map.


In [ ]:
class UncertaintyHead(nn.Module):
    def __init__(self, in_channels, dropout_p=0.3):
        super().__init__()
        self.dropout = nn.Dropout2d(dropout_p)
        self.conv = nn.Conv2d(in_channels, 1, 1)

    def forward(self, x):
        return torch.sigmoid(self.conv(self.dropout(x)))

@torch.no_grad()
def mc_inference(model, x_vv, x_vh, terrain, n_passes=10):
    was_training = model.training
    model.train()
    preds = torch.stack([model(x_vv, x_vh, terrain) for _ in range(n_passes)], dim=0)
    mean_pred = preds.mean(0)
    uncertainty = preds.std(0)
    model.train(was_training)
    return mean_pred, uncertainty


## Section 5.3.4 — Full `CASANet` Class

CASA-Net combines asymmetric VV/VH encoding, CPAG fusion, terrain-conditioned decoding, and optional MC Dropout. The same class is also used for the CPAG-only and HTC-only variants by switching components on or off.


In [ ]:
class ResNet34Encoder(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        backbone = torchvision.models.resnet34(weights=torchvision.models.ResNet34_Weights.DEFAULT)
        old = backbone.conv1
        backbone.conv1 = nn.Conv2d(in_channels, old.out_channels, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            backbone.conv1.weight[:] = old.weight.mean(dim=1, keepdim=True)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(self, x):
        x = self.stem(x)
        f1 = self.layer1(self.maxpool(x))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return [f1, f2, f3, f4]

class MobileNetV3Encoder(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        backbone = torchvision.models.mobilenet_v3_small(weights=torchvision.models.MobileNet_V3_Small_Weights.DEFAULT)
        old = backbone.features[0][0]
        backbone.features[0][0] = nn.Conv2d(in_channels, old.out_channels, kernel_size=old.kernel_size, stride=old.stride, padding=old.padding, bias=False)
        with torch.no_grad():
            backbone.features[0][0].weight[:] = old.weight.mean(dim=1, keepdim=True)
        self.features = backbone.features

    def forward(self, x):
        outs = []
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if idx in {1, 3, 6, 11}:
                outs.append(x)
        return outs

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = F.relu(self.bn2(self.conv2(x)), inplace=True)
        return x

class UNetDecoder(nn.Module):
    def __init__(self, use_htc=True):
        super().__init__()
        self.use_htc = use_htc
        self.blocks = nn.ModuleList([
            DecoderBlock(512, 256, 256),
            DecoderBlock(256, 128, 128),
            DecoderBlock(128, 64, 64),
        ])
        self.final = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))

    def forward(self, fused, film_params=None):
        x = self.blocks[0](fused[-1], fused[-2]); x = apply_film(x, film_params[0]) if self.use_htc and film_params else x
        x = self.blocks[1](x, fused[-3]); x = apply_film(x, film_params[1]) if self.use_htc and film_params else x
        x = self.blocks[2](x, fused[-4]); x = apply_film(x, film_params[2]) if self.use_htc and film_params else x
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = self.final(x)
        x = apply_film(x, film_params[3]) if self.use_htc and film_params else x
        return x

class CASANet(nn.Module):
    def __init__(self, dropout_p=0.3, use_cpag=True, use_htc=True, use_mc_dropout=True):
        super().__init__()
        self.use_cpag = use_cpag
        self.use_htc = use_htc
        self.vv_encoder = ResNet34Encoder(1)
        self.vh_encoder = MobileNetV3Encoder(1)
        self.vh_proj = nn.ModuleList([nn.Conv2d(16, 64, 1), nn.Conv2d(24, 128, 1), nn.Conv2d(48, 256, 1), nn.Conv2d(96, 512, 1)])
        self.cpag = nn.ModuleList([CPAG(64, 16, 32), CPAG(128, 24, 64), CPAG(256, 48, 128), CPAG(512, 96, 256)])
        self.terrain_branch = HaorTerrainBranch()
        self.decoder = UNetDecoder(use_htc=use_htc)
        self.head = UncertaintyHead(32, dropout_p if use_mc_dropout else 0.0)

    def forward(self, x_vv, x_vh, x_terrain):
        vv_feats = self.vv_encoder(x_vv)
        vh_feats = self.vh_encoder(x_vh)
        fused = [self.cpag[i](vv_feats[i], vh_feats[i]) if self.use_cpag else vv_feats[i] + self.vh_proj[i](vh_feats[i]) for i in range(4)]
        film_params = self.terrain_branch(x_terrain) if self.use_htc else None
        return self.head(self.decoder(fused, film_params))

probe = CASANet()
param_summary = pd.DataFrame([
    {'component': 'VV encoder (ResNet34)', 'params': sum(p.numel() for p in probe.vv_encoder.parameters())},
    {'component': 'VH encoder (MobileNetV3)', 'params': sum(p.numel() for p in probe.vh_encoder.parameters())},
    {'component': 'CPAG modules (4)', 'params': sum(p.numel() for p in probe.cpag.parameters())},
    {'component': 'Terrain branch', 'params': sum(p.numel() for p in probe.terrain_branch.parameters())},
    {'component': 'Decoder', 'params': sum(p.numel() for p in probe.decoder.parameters())},
    {'component': 'Total', 'params': sum(p.numel() for p in probe.parameters())},
])
display(param_summary)


## Section 5.3.5 — `BCEDiceLoss`

BCEDiceLoss balances pixelwise calibration with overlap-sensitive segmentation quality, which is a strong fit for imbalanced flood masks.


In [ ]:
class BCEDiceLoss(nn.Module):
    def forward(self, pred, target):
        bce = F.binary_cross_entropy(pred, target)
        smooth = 1e-6
        intersection = (pred * target).sum()
        dice = 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)
        return 0.5 * bce + 0.5 * dice


## Section 5.4 ? Model Benchmark + CASA-Net Ablation

This section now does two things:

1. Benchmarks CASA-Net against five established segmentation baselines.
2. Preserves the internal CASA-Net ablation so the contribution of CPAG, HTC, and MC Dropout is still visible.

The five comparison models are:
- U-Net
- U-Net++
- FPN
- DeepLabV3+
- MAnet (attention-based baseline)

CASA-Net is evaluated alongside them under the same optimizer, scheduler, split, and early-stopping rule.


In [ ]:
def tensor_iou(pred, target, threshold=0.5):
    pred_bin = (pred >= threshold).float()
    target_bin = (target >= 0.5).float()
    intersection = (pred_bin * target_bin).sum().item()
    union = pred_bin.sum().item() + target_bin.sum().item() - intersection
    return intersection / max(union, 1.0)

def tensor_dice(pred, target, threshold=0.5):
    pred_bin = (pred >= threshold).float()
    target_bin = (target >= 0.5).float()
    intersection = (pred_bin * target_bin).sum().item()
    return (2 * intersection) / max(pred_bin.sum().item() + target_bin.sum().item(), 1.0)

class BaselineWrapper(nn.Module):
    def __init__(self, arch_name):
        super().__init__()
        if smp is None:
            raise ImportError('segmentation_models_pytorch is required for benchmark baselines')
        builders = {
            'U-Net': lambda: smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=2, classes=1, activation='sigmoid'),
            'U-Net++': lambda: smp.UnetPlusPlus(encoder_name='resnet34', encoder_weights='imagenet', in_channels=2, classes=1, activation='sigmoid'),
            'FPN': lambda: smp.FPN(encoder_name='resnet34', encoder_weights='imagenet', in_channels=2, classes=1, activation='sigmoid'),
            'DeepLabV3+': lambda: smp.DeepLabV3Plus(encoder_name='resnet34', encoder_weights='imagenet', in_channels=2, classes=1, activation='sigmoid'),
            'MAnet': lambda: smp.MAnet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=2, classes=1, activation='sigmoid'),
        }
        self.arch_name = arch_name
        self.model = builders[arch_name]()

    def forward(self, x_vv, x_vh, x_terrain):
        return self.model(torch.cat([x_vv, x_vh], dim=1))

def evaluate_epoch(model, loader, criterion):
    model.eval()
    losses, ious, dices = [], [], []
    with torch.no_grad():
        for vv, vh, terrain, mask in loader:
            vv, vh, terrain, mask = vv.to(DEVICE).float(), vh.to(DEVICE).float(), terrain.to(DEVICE).float(), mask.to(DEVICE).float()
            pred = model(vv, vh, terrain)
            loss = criterion(pred, mask)
            losses.append(loss.item())
            ious.append(tensor_iou(pred, mask))
            dices.append(tensor_dice(pred, mask))
    return {'loss': float(np.mean(losses)), 'iou': float(np.mean(ious)), 'dice': float(np.mean(dices)), 'f1': float(np.mean(dices))}

def train_variant(name, model, checkpoint_path, epochs=50, patience=10):
    model = model.to(DEVICE)
    criterion = BCEDiceLoss()
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    history, best_state, best_dice, wait = [], None, -1.0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for vv, vh, terrain, mask in tqdm(train_loader, desc=f'{name} epoch {epoch}/{epochs}', leave=False):
            vv, vh, terrain, mask = vv.to(DEVICE).float(), vh.to(DEVICE).float(), terrain.to(DEVICE).float(), mask.to(DEVICE).float()
            optimizer.zero_grad()
            pred = model(vv, vh, terrain)
            loss = criterion(pred, mask)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()
        val_metrics = evaluate_epoch(model, val_loader, criterion)
        history.append({'epoch': epoch, 'train_loss': float(np.mean(train_losses)), 'val_loss': val_metrics['loss'], 'val_iou': val_metrics['iou'], 'val_dice': val_metrics['dice'], 'val_f1': val_metrics['f1']})
        if val_metrics['dice'] > best_dice:
            best_dice = val_metrics['dice']
            best_state = deepcopy(model.state_dict())
            torch.save(best_state, checkpoint_path)
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break
    history_df = pd.DataFrame(history)
    history_df.to_csv(REPORT_DIR / f"{name.lower().replace(' ', '_').replace('+', 'plus').replace('-', '').replace('/', '_')}_history.csv", index=False)
    return best_state, history_df

benchmark_variants = {
    'U-Net': BaselineWrapper('U-Net'),
    'U-Net++': BaselineWrapper('U-Net++'),
    'FPN': BaselineWrapper('FPN'),
    'DeepLabV3+': BaselineWrapper('DeepLabV3+'),
    'MAnet': BaselineWrapper('MAnet'),
    'CASA-Net': CASANet(dropout_p=0.3, use_cpag=True, use_htc=True, use_mc_dropout=True),
}

ablation_variants = {
    'Baseline U-Net': BaselineWrapper('U-Net'),
    '+CPAG': CASANet(dropout_p=0.0, use_cpag=True, use_htc=False, use_mc_dropout=False),
    '+HTC': CASANet(dropout_p=0.0, use_cpag=False, use_htc=True, use_mc_dropout=False),
    'CASA-Net (full)': CASANet(dropout_p=0.3, use_cpag=True, use_htc=True, use_mc_dropout=True),
}

benchmark_checkpoints = {
    'U-Net': MODELS_DIR / 'benchmark_unet.pth',
    'U-Net++': MODELS_DIR / 'benchmark_unetplusplus.pth',
    'FPN': MODELS_DIR / 'benchmark_fpn.pth',
    'DeepLabV3+': MODELS_DIR / 'benchmark_deeplabv3plus.pth',
    'MAnet': MODELS_DIR / 'benchmark_manet.pth',
    'CASA-Net': MODELS_DIR / 'casa_net_best.pth',
}

ablation_checkpoints = {
    'Baseline U-Net': MODELS_DIR / 'baseline_unet.pth',
    '+CPAG': MODELS_DIR / 'cpag_only.pth',
    '+HTC': MODELS_DIR / 'htc_only.pth',
    'CASA-Net (full)': MODELS_DIR / 'casa_net_best.pth',
}

benchmark_rows, ablation_rows = [], []
benchmark_histories, ablation_histories = {}, {}

for name, model in benchmark_variants.items():
    best_state, history_df = train_variant(name, model, benchmark_checkpoints[name])
    benchmark_histories[name] = history_df
    model.load_state_dict(best_state)
    metrics = evaluate_epoch(model.to(DEVICE), val_loader, BCEDiceLoss())
    benchmark_rows.append({'Model': name, 'Val IoU': metrics['iou'], 'Val F1': metrics['f1'], 'Val Dice': metrics['dice'], 'Params': sum(p.numel() for p in model.parameters())})

for name, model in ablation_variants.items():
    best_state, history_df = train_variant(name, model, ablation_checkpoints[name])
    ablation_histories[name] = history_df
    model.load_state_dict(best_state)
    metrics = evaluate_epoch(model.to(DEVICE), val_loader, BCEDiceLoss())
    ablation_rows.append({'Model': name, 'Val IoU': metrics['iou'], 'Val F1': metrics['f1'], 'Val Dice': metrics['dice'], 'Params': sum(p.numel() for p in model.parameters()), 'Notes': {'Baseline U-Net': 'VV+VH concat', '+CPAG': 'Cross-pol attention', '+HTC': 'Terrain conditioning', 'CASA-Net (full)': 'CPAG + HTC + MC Dropout'}[name]})

benchmark_df = pd.DataFrame(benchmark_rows).sort_values(['Val IoU', 'Val F1', 'Val Dice'], ascending=False).reset_index(drop=True)
benchmark_df['Rank'] = np.arange(1, len(benchmark_df) + 1)
benchmark_df.to_csv(REPORT_DIR / 'benchmark_results.csv', index=False)

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(REPORT_DIR / 'ablation_results.csv', index=False)

display(benchmark_df)
display(ablation_df)
if not benchmark_df.empty:
    print('Top benchmark model:', benchmark_df.iloc[0]['Model'])


## Section 5.5 ? Benchmark and Ablation Results Tables

The first table compares CASA-Net against five external baselines. The second table keeps the internal ablation story intact so both competitiveness and architectural contribution are visible.


In [ ]:
display(benchmark_df.style.format({'Val IoU': '{:.4f}', 'Val F1': '{:.4f}', 'Val Dice': '{:.4f}', 'Params': '{:,}'}))
display(ablation_df.style.format({'Val IoU': '{:.4f}', 'Val F1': '{:.4f}', 'Val Dice': '{:.4f}', 'Params': '{:,}'}))


## Section 5.6 ? Training Curves and Evaluation Diagrams

This notebook now exports a fuller benchmark package:

1. `benchmark_model_comparison.png`
2. `benchmark_ranked_leaderboard.png`
3. `benchmark_radar_chart.png`
4. `ablation_training_curves.png`
5. `ablation_metrics_bar.png`
6. `confusion_matrix_casa_net.png`
7. `precision_recall_roc_casa_net.png`
8. `uncertainty_sample_patches.png`

These are the main model-comparison visuals, with scene-level figures still produced in Notebooks 06 and 08.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), dpi=300)
pos = np.arange(len(benchmark_df))
ax.bar(pos - 0.25, benchmark_df['Val IoU'], width=0.25, label='IoU', color='#264653')
ax.bar(pos, benchmark_df['Val F1'], width=0.25, label='F1', color='#2a9d8f')
ax.bar(pos + 0.25, benchmark_df['Val Dice'], width=0.25, label='Dice', color='#e76f51')
ax.set_xticks(pos)
ax.set_xticklabels(benchmark_df['Model'], rotation=15)
ax.set_ylim(0, 1)
ax.set_title('CASA-Net vs 5 benchmark models')
ax.legend()
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'benchmark_model_comparison.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
leaderboard = benchmark_df.sort_values('Rank')
colors = ['#d62828' if m == 'CASA-Net' else '#577590' for m in leaderboard['Model']]
ax.barh(leaderboard['Model'], leaderboard['Val IoU'], color=colors)
ax.invert_yaxis()
ax.set_xlabel('Validation IoU')
ax.set_title('Benchmark leaderboard')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'benchmark_ranked_leaderboard.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

categories = ['Val IoU', 'Val F1', 'Val Dice']
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]
fig = plt.figure(figsize=(7, 7), dpi=300)
ax = plt.subplot(111, polar=True)
for _, row in benchmark_df.iterrows():
    values = [row[c] for c in categories]
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=row['Model'])
    if row['Model'] == 'CASA-Net':
        ax.fill(angles, values, alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_title('Benchmark radar chart')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'benchmark_radar_chart.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
for ax, (name, history_df) in zip(axes.ravel(), ablation_histories.items()):
    ax.plot(history_df['epoch'], history_df['val_iou'], color='#1d3557', linewidth=2)
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Val IoU')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'ablation_training_curves.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
pos = np.arange(len(ablation_df))
ax.bar(pos - 0.2, ablation_df['Val IoU'], width=0.2, label='IoU', color='#264653')
ax.bar(pos, ablation_df['Val F1'], width=0.2, label='F1', color='#2a9d8f')
ax.bar(pos + 0.2, ablation_df['Val Dice'], width=0.2, label='Dice', color='#e76f51')
ax.set_xticks(pos); ax.set_xticklabels(ablation_df['Model'], rotation=15); ax.set_ylim(0, 1); ax.legend(); ax.set_title('CASA-Net ablation metrics comparison')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'ablation_metrics_bar.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)


## Section 5.7 — Test Set Evaluation (CASA-Net Full)

The held-out evaluation summarizes the final model’s performance and exports the confusion matrix plus the precision-recall/ROC diagnostic figure. These become two of the core evaluation diagrams for the submission package.


In [ ]:
full_model = CASANet(dropout_p=0.3, use_cpag=True, use_htc=True, use_mc_dropout=True).to(DEVICE)
full_model.load_state_dict(torch.load(MODELS_DIR / 'casa_net_best.pth', map_location=DEVICE))
full_model.eval()
all_probs = []; all_unc = []; all_targets = []
with torch.no_grad():
    for vv, vh, terrain, mask in tqdm(test_loader, desc='Testing CASA-Net'):
        vv, vh, terrain, mask = vv.to(DEVICE).float(), vh.to(DEVICE).float(), terrain.to(DEVICE).float(), mask.to(DEVICE).float()
        mean_pred, uncertainty = mc_inference(full_model, vv, vh, terrain, n_passes=10)
        all_probs.append(mean_pred.cpu().numpy().ravel())
        all_unc.append(uncertainty.cpu().numpy().ravel())
        all_targets.append(mask.cpu().numpy().ravel())
y_prob = np.concatenate(all_probs); y_unc = np.concatenate(all_unc); y_true = np.concatenate(all_targets); y_pred = (y_prob >= 0.5).astype(np.uint8)
intersection = np.logical_and(y_true == 1, y_pred == 1).sum(); union = np.logical_or(y_true == 1, y_pred == 1).sum()
metrics_df = pd.DataFrame([{'Accuracy': accuracy_score(y_true, y_pred), 'Precision': precision_score(y_true, y_pred, zero_division=0), 'Recall': recall_score(y_true, y_pred, zero_division=0), 'F1': f1_score(y_true, y_pred, zero_division=0), 'IoU': intersection / max(union, 1), 'Dice': (2 * intersection) / max((y_true == 1).sum() + (y_pred == 1).sum(), 1)}])
metrics_df.to_csv(REPORT_DIR / 'test_metrics_casa_net.csv', index=False)
display(metrics_df)
print(f"Target IoU > 0.75, F1 > 0.83 -> Achieved: IoU={metrics_df.loc[0, 'IoU']:.4f}, F1={metrics_df.loc[0, 'F1']:.4f}")
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4), dpi=300); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax); ax.set_title('CASA-Net confusion matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); plt.tight_layout(); plt.savefig(FIGURES_DIR / 'confusion_matrix_casa_net.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)
precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob); fpr, tpr, _ = roc_curve(y_true, y_prob)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=300)
axes[0].plot(recall_vals, precision_vals, color='#1d3557', linewidth=2); axes[0].set_title('Precision-Recall'); axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[1].plot(fpr, tpr, color='#e76f51', linewidth=2, label=f"AUC={auc(fpr, tpr):.3f}"); axes[1].plot([0, 1], [0, 1], '--', color='gray'); axes[1].legend(); axes[1].set_title('ROC'); axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'precision_recall_roc_casa_net.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)


## Section 5.8 — Uncertainty Analysis

The uncertainty plot highlights where the model is confident and where it is ambiguous. Those high-uncertainty areas are carried into Notebook 07 for uncertainty-aware facility flagging.


In [ ]:
indices = [0, min(1, len(test_ds) - 1), min(2, len(test_ds) - 1)]
fig, axes = plt.subplots(len(indices), 3, figsize=(12, 10), dpi=300)
if len(indices) == 1:
    axes = np.expand_dims(axes, axis=0)
for row_idx, sample_idx in enumerate(indices):
    vv, vh, terrain, mask = test_ds[sample_idx]
    mean_pred, uncertainty = mc_inference(full_model, vv.unsqueeze(0).to(DEVICE).float(), vh.unsqueeze(0).to(DEVICE).float(), terrain.unsqueeze(0).to(DEVICE).float(), n_passes=10)
    axes[row_idx, 0].imshow(vv.numpy()[0], cmap='gray'); axes[row_idx, 0].set_title('VV input')
    axes[row_idx, 1].imshow(mean_pred.cpu().numpy()[0, 0], cmap='Blues', vmin=0, vmax=1); axes[row_idx, 1].set_title('Flood probability')
    axes[row_idx, 2].imshow(uncertainty.cpu().numpy()[0, 0], cmap='magma'); axes[row_idx, 2].set_title('Epistemic uncertainty')
    for ax in axes[row_idx]:
        ax.axis('off')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'uncertainty_sample_patches.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)
print('Highest expected uncertainty: haor permanent-water boundaries, paddy-field edges, and urban double-bounce zones.')


<!-- MULTI-HAZARD EXTENSION GENERATED -->
## Section 5.9 ? Multi-Hazard CASA-Net Extension
This section extends CASA-Net into a **shared-backbone, multi-head model**. Flood remains the validated benchmark branch, while erosion and landslide now use the initial seed supervision assets bundled in the multi-hazard patch set.


In [ ]:
# MULTI-HAZARD EXTENSION GENERATED
from analysis.multi_hazard_support import (
    load_hazard_catalog,
    ordered_hazard_ids,
    build_hazard_target_stack,
    MultiHazardProbabilityHeads,
    multi_hazard_bcedice_loss,
)

ROOT = Path(r'f:\MAPATHON\sylhet_flood_2024')
hazard_catalog = load_hazard_catalog(ROOT / 'config' / 'hazard_catalog.json')
hazard_names = ordered_hazard_ids(hazard_catalog)
hazard_weights = {haz['id']: haz['hazard_weight'] for haz in hazard_catalog['hazards']}

class MultiHazardPatchDataset(Dataset):
    def __init__(self, patch_dir=ROOT / 'data' / 'processed' / 'patches_multihazard_ready', split='train', augment=False):
        self.patch_dir = Path(patch_dir)
        self.split = split
        self.augment = augment
        self.split_index = json.loads((self.patch_dir / 'split_index.json').read_text()) if (self.patch_dir / 'split_index.json').exists() else {split: []}
        self.paths = [Path(p) for p in self.split_index.get(split, [])]
        self.norm = json.loads((self.patch_dir / 'norm_stats.json').read_text()) if (self.patch_dir / 'norm_stats.json').exists() else {'vv_mean': 0.0, 'vv_std': 1.0, 'vh_mean': 0.0, 'vh_std': 1.0}

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with np.load(self.paths[idx]) as npz:
            vv = normalize(npz['vv'].astype(np.float32), self.norm['vv_mean'], self.norm['vv_std'])
            vh = normalize(npz['vh'].astype(np.float32), self.norm['vh_mean'], self.norm['vh_std'])
            terrain = npz['terrain'].astype(np.float32)
            hazard_targets = build_hazard_target_stack(npz, hazard_names)
        return (
            torch.tensor(vv, dtype=torch.float32),
            torch.tensor(vh, dtype=torch.float32),
            torch.tensor(terrain, dtype=torch.float32),
            torch.tensor(hazard_targets, dtype=torch.float32),
        )

class MultiHazardCASANet(nn.Module):
    def __init__(self, hazard_names=hazard_names, dropout_p=0.2):
        super().__init__()
        self.hazard_names = list(hazard_names)
        self.vv_encoder = ResNet34Encoder(in_channels=1)
        self.vh_encoder = MobileNetV3Encoder(in_channels=1)
        self.cpag = nn.ModuleList([
            CPAG(vv_ch, vh_ch, inter_ch)
            for vv_ch, vh_ch, inter_ch in [
                (64, 16, 32),
                (128, 24, 64),
                (256, 48, 128),
                (512, 96, 256),
            ]
        ])
        self.terrain_branch = HaorTerrainBranch(in_channels=4, decoder_channels=[256, 128, 64, 32])
        self.decoder = UNetDecoder(encoder_channels=[512, 256, 128, 64], decoder_channels=[256, 128, 64, 32])
        self.hazard_heads = MultiHazardProbabilityHeads(32, self.hazard_names, dropout_p=dropout_p)

    def forward(self, x_vv, x_vh, x_terrain):
        vv_feats = self.vv_encoder(x_vv)
        vh_feats = self.vh_encoder(x_vh)
        fused = [self.cpag[i](vv_feats[i], vh_feats[i]) for i in range(4)]
        film_params = self.terrain_branch(x_terrain)
        decoded = self.decoder(fused, film_params)
        return self.hazard_heads(decoded)

def multi_hazard_loss(predictions, targets):
    return multi_hazard_bcedice_loss(predictions, targets, hazard_names, hazard_weights)

multi_hazard_training_plan = pd.DataFrame([
    {'hazard': hazard['id'], 'hazard_weight': hazard['hazard_weight'], 'task': hazard['task'], 'supervision_expected': hazard['id'] == 'flood'}
    for hazard in hazard_catalog['hazards']
])
multi_hazard_training_plan.to_csv(ROOT / 'outputs' / 'report' / 'multi_hazard_training_plan.csv', index=False)
multi_hazard_training_plan
